# The MCP Case for Yahoo Finance — Transparent Colab Laboratory

**Companion notebook to the paper by Alejandro Reynoso**  
*Building a Governed Market-Data Server for Agentic Research, Dynamic Memory, and Full Auditability*

This notebook makes the complete architecture inspectable and executable. It builds a modular MCP server inside the Colab runtime, tests it first with deterministic simulated evidence, optionally calls Yahoo-origin data through the unofficial `yfinance` adapter, writes an Obsidian-compatible dynamic vault, and creates a cryptographically verifiable audit bundle.

> **Important classification.** Yahoo does not supply the MCP server. We build the server. `yfinance` is an unofficial interface; observations obtained through it are labeled `DELAYED_OR_LIVE_UNVERIFIED`, never automatically `LIVE`. This notebook is for research and teaching, not trading or investment advice.

## Learning objectives and paper map

By completing the notebook, the reader can:

1. distinguish the model, MCP client, MCP server, provider adapter, scheduler, vault, and audit layer;
2. inspect how Python annotations and docstrings become discoverable MCP tool contracts;
3. replace Yahoo without changing the model-facing tools;
4. apply authorization, universe, parameter, freshness, completeness, and classification gates;
5. produce path-dependent reports from persistent predecessor memory;
6. verify lineage and SHA-256 integrity; and
7. separate a classroom prototype from a production market-data service.

The sections follow the paper: problem → capability boundary → internal architecture → lifecycle → implementation → governance → agency → dynamic vault → audit → tests → deployment.

## 1. Architectural separation

```text
User objective
      ↓
Reasoning host / agent ── scheduler supplies time
      ↓
MCP client → MCP server → governance → market service → provider adapter → Yahoo/yfinance
      ↑                         ↓
dynamic vault ← reports ← canonical observations → audit events and manifest
```

MCP supplies a bounded capability contract. It does **not** supply the clock, data rights, judgment, persistence, or accountability.

In [1]:
# 2. Install only the libraries required by this notebook.
import sys, subprocess

packages = ["mcp>=1.9,<2", "yfinance>=0.2.50,<2", "pandas>=2,<3"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependencies installed.")

Dependencies installed.


In [2]:
# 3. Select a transparent working location.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Set USE_DRIVE=True if you want durable files in MyDrive.
USE_DRIVE = False
if IN_COLAB and USE_DRIVE:
    WORK_ROOT = Path("/content/drive/MyDrive/THE ESSENTIAL AUTONOMOUS WORKFLOWS/MCP ESSENTIALS")
else:
    WORK_ROOT = Path("/content/Yahoo_Finance_MCP_Case") if IN_COLAB else Path.cwd() / "Yahoo_Finance_MCP_Case"

PROJECT_ROOT = WORK_ROOT / "yahoo_market_mcp"
VAULT_ROOT = WORK_ROOT / "Yahoo_Market_Obsidian_Vault"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
VAULT_ROOT.mkdir(parents=True, exist_ok=True)
print({"project": str(PROJECT_ROOT), "vault": str(VAULT_ROOT), "persistent_drive": bool(IN_COLAB and USE_DRIVE)})

Mounted at /content/drive
{'project': '/content/Yahoo_Finance_MCP_Case/yahoo_market_mcp', 'vault': '/content/Yahoo_Finance_MCP_Case/Yahoo_Market_Obsidian_Vault', 'persistent_drive': False}


## 4. Build the domain and provider layer

The canonical `Observation` is deliberately independent of Yahoo. Both the simulated and Yahoo adapters return the same object. That makes the tool contract stable when the provider changes.

In [3]:
# 4A. Write domain.py: canonical data model and approved universe.
domain_py = r'''from dataclasses import asdict, dataclass
from typing import Optional

APPROVED_UNIVERSE = {
    "^GSPC": "S&P 500",
    "^DJI": "Dow Jones Industrial Average",
    "^IXIC": "Nasdaq Composite",
    "^FTSE": "FTSE 100",
    "^N225": "Nikkei 225",
}

@dataclass(frozen=True)
class Observation:
    symbol: str
    name: str
    value: float
    change_pct: float
    volume: Optional[float]
    observation_timestamp: str
    retrieved_at: str
    provider_adapter: str
    source_family: str
    classification: str

    def to_dict(self):
        return asdict(self)
'''
(PROJECT_ROOT / "domain.py").write_text(domain_py, encoding="utf-8")
print(domain_py)

from dataclasses import asdict, dataclass
from typing import Optional

APPROVED_UNIVERSE = {
    "^GSPC": "S&P 500",
    "^DJI": "Dow Jones Industrial Average",
    "^IXIC": "Nasdaq Composite",
    "^FTSE": "FTSE 100",
    "^N225": "Nikkei 225",
}

@dataclass(frozen=True)
class Observation:
    symbol: str
    name: str
    value: float
    change_pct: float
    volume: Optional[float]
    observation_timestamp: str
    retrieved_at: str
    provider_adapter: str
    source_family: str
    classification: str

    def to_dict(self):
        return asdict(self)



In [4]:
# 4B. Write providers.py: provider protocol, deterministic adapter, and Yahoo adapter.
providers_py = r'''from datetime import datetime, timezone
from typing import Protocol
import math

from domain import APPROVED_UNIVERSE, Observation

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def number(value):
    if value is None:
        return None
    value = float(value)
    return None if math.isnan(value) else value

class MarketProvider(Protocol):
    name: str
    def get_quote(self, symbol: str) -> Observation: ...
    def get_history(self, symbol: str, period: str, interval: str) -> list[dict]: ...

class SimulatedProvider:
    name = "deterministic-classroom-adapter"
    _levels = {"^GSPC": 6300, "^DJI": 45000, "^IXIC": 21000, "^FTSE": 9000, "^N225": 41500}
    _changes = {"^GSPC": .31, "^DJI": .08, "^IXIC": .68, "^FTSE": -.22, "^N225": -.35}

    def get_quote(self, symbol):
        if symbol not in APPROVED_UNIVERSE:
            raise ValueError(f"Unknown symbol: {symbol}")
        t = now_utc(); change = self._changes[symbol]
        return Observation(symbol, APPROVED_UNIVERSE[symbol],
            round(self._levels[symbol] * (1 + change / 100), 4), change, None,
            t, t, self.name, "Synthetic classroom data", "SIMULATED")

    def get_history(self, symbol, period, interval):
        q = self.get_quote(symbol)
        return [{"timestamp": q.observation_timestamp, "open": q.value/(1+q.change_pct/100),
                 "high": q.value*1.001, "low": q.value*.999, "close": q.value, "volume": None}]

class YFinanceProvider:
    name = "yfinance-unofficial-yahoo-adapter"

    def _authorize_symbol(self, symbol):
        if symbol not in APPROVED_UNIVERSE:
            raise ValueError(f"Symbol is not authorized: {symbol}")

    def get_quote(self, symbol):
        self._authorize_symbol(symbol)
        import yfinance as yf
        frame = yf.Ticker(symbol).history(period="2d", interval="1m", auto_adjust=False)
        frame = frame.dropna(subset=["Close"])
        if frame.empty:
            raise RuntimeError(f"No observations returned for {symbol}")
        latest, baseline = frame.iloc[-1], frame.iloc[0]
        value, opening = number(latest["Close"]), number(baseline["Close"])
        if value is None or opening in (None, 0):
            raise RuntimeError(f"Invalid value or baseline for {symbol}")
        return Observation(symbol, APPROVED_UNIVERSE[symbol], round(value, 4),
            round((value/opening-1)*100, 4), number(latest.get("Volume")),
            frame.index[-1].isoformat(), now_utc(), self.name, "Yahoo Finance",
            "DELAYED_OR_LIVE_UNVERIFIED")

    def get_history(self, symbol, period, interval):
        self._authorize_symbol(symbol)
        import yfinance as yf
        frame = yf.Ticker(symbol).history(period=period, interval=interval, auto_adjust=False)
        if frame.empty:
            raise RuntimeError(f"No history returned for {symbol}")
        return [{"timestamp": ts.isoformat(), "open": number(r["Open"]),
                 "high": number(r["High"]), "low": number(r["Low"]),
                 "close": number(r["Close"]), "volume": number(r.get("Volume"))}
                for ts, r in frame.iterrows()]
'''
(PROJECT_ROOT / "providers.py").write_text(providers_py, encoding="utf-8")
print("Provider layer written.")

Provider layer written.


## 5. Governance before retrieval and publication

The policy applies at two moments:

- **pre-retrieval:** authorization, approved universe, bounded history parameters;
- **post-retrieval:** provenance, classification, completeness, plausible values, timestamp presence, and freshness qualification.

A rejection becomes evidence. It is never silently dropped.

In [5]:
# 5A. Write governance.py.
governance_py = r'''from datetime import datetime, timezone
from domain import APPROVED_UNIVERSE

PERMITTED_PERIODS = {"1d", "5d", "1mo", "3mo"}
PERMITTED_INTERVALS = {"1m", "5m", "15m", "1h", "1d"}
PERMITTED_CLASSES = {"SIMULATED", "DELAYED", "LIVE", "DELAYED_OR_LIVE_UNVERIFIED"}

class GovernanceEngine:
    def __init__(self, authorization="RESEARCH-AUTHORIZED", max_age_seconds=900):
        self.authorization = authorization
        self.max_age_seconds = max_age_seconds

    def authorize_request(self, token, symbol=None, period=None, interval=None):
        gates = {"authorization": token == self.authorization}
        if symbol is not None: gates["universe"] = symbol in APPROVED_UNIVERSE
        if period is not None: gates["period"] = period in PERMITTED_PERIODS
        if interval is not None: gates["interval"] = interval in PERMITTED_INTERVALS
        return self._decision(gates, "pre_retrieval")

    def validate_observations(self, observations, requested_symbols):
        observed = {o.symbol for o in observations}
        gates = {
            "completeness": observed == set(requested_symbols),
            "positive_values": all(o.value > 0 for o in observations),
            "provenance": all(o.provider_adapter and o.source_family for o in observations),
            "timestamps": all(o.observation_timestamp and o.retrieved_at for o in observations),
            "classification": all(o.classification in PERMITTED_CLASSES for o in observations),
        }
        # We preserve the provider timestamp and do not pretend that unverified data is live.
        return self._decision(gates, "post_retrieval")

    def _decision(self, gates, stage):
        return {"stage": stage, "accepted": all(gates.values()), "gates": gates,
                "evaluated_at": datetime.now(timezone.utc).isoformat()}
'''
(PROJECT_ROOT / "governance.py").write_text(governance_py, encoding="utf-8")
print(governance_py)

from datetime import datetime, timezone
from domain import APPROVED_UNIVERSE

PERMITTED_PERIODS = {"1d", "5d", "1mo", "3mo"}
PERMITTED_INTERVALS = {"1m", "5m", "15m", "1h", "1d"}
PERMITTED_CLASSES = {"SIMULATED", "DELAYED", "LIVE", "DELAYED_OR_LIVE_UNVERIFIED"}

class GovernanceEngine:
    def __init__(self, authorization="RESEARCH-AUTHORIZED", max_age_seconds=900):
        self.authorization = authorization
        self.max_age_seconds = max_age_seconds

    def authorize_request(self, token, symbol=None, period=None, interval=None):
        gates = {"authorization": token == self.authorization}
        if symbol is not None: gates["universe"] = symbol in APPROVED_UNIVERSE
        if period is not None: gates["period"] = period in PERMITTED_PERIODS
        if interval is not None: gates["interval"] = interval in PERMITTED_INTERVALS
        return self._decision(gates, "pre_retrieval")

    def validate_observations(self, observations, requested_symbols):
        observed = {o.symbol f

## 6. Market service: the application boundary

The service coordinates authorization, provider calls, canonical output, partial failures, and audit recording. The MCP handlers remain thin.

In [6]:
# 6A. Write audit.py and service.py.
audit_py = r'''import json
from datetime import datetime, timezone
from pathlib import Path

class AuditLog:
    def __init__(self, path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
    def write(self, event):
        payload = {"recorded_at": datetime.now(timezone.utc).isoformat(), **event}
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(payload, default=str) + "\n")
        return payload
'''
service_py = r'''from domain import APPROVED_UNIVERSE

class MarketService:
    def __init__(self, provider, governance, audit):
        self.provider, self.governance, self.audit = provider, governance, audit

    def universe(self):
        return {"symbols": APPROVED_UNIVERSE, "count": len(APPROVED_UNIVERSE),
                "provider_adapter": self.provider.name}

    def quote(self, symbol, authorization):
        pre = self.governance.authorize_request(authorization, symbol=symbol)
        self.audit.write({"event": "PRE_RETRIEVAL", "tool": "get_quote", "symbol": symbol, "decision": pre})
        if not pre["accepted"]: raise PermissionError(f"Request rejected: {pre['gates']}")
        observation = self.provider.get_quote(symbol)
        post = self.governance.validate_observations([observation], [symbol])
        self.audit.write({"event": "POST_RETRIEVAL", "tool": "get_quote", "symbol": symbol, "decision": post})
        if not post["accepted"]: raise RuntimeError(f"Observation rejected: {post['gates']}")
        return {"observation": observation.to_dict(), "governance": {"pre": pre, "post": post}}

    def snapshot(self, authorization):
        pre = self.governance.authorize_request(authorization)
        self.audit.write({"event": "PRE_RETRIEVAL", "tool": "get_market_snapshot", "decision": pre})
        if not pre["accepted"]: raise PermissionError(f"Request rejected: {pre['gates']}")
        observations, errors = [], []
        for symbol, name in APPROVED_UNIVERSE.items():
            try: observations.append(self.provider.get_quote(symbol))
            except Exception as exc:
                errors.append({"symbol": symbol, "name": name, "error_type": type(exc).__name__, "error": str(exc)})
        post = self.governance.validate_observations(observations, APPROVED_UNIVERSE)
        status = "ACCEPTED" if post["accepted"] else "QUALIFIED_OR_REJECTED"
        result = {"status": status, "requested_count": len(APPROVED_UNIVERSE),
                  "successful_count": len(observations), "complete": not errors,
                  "observations": [o.to_dict() for o in observations], "errors": errors,
                  "governance": {"pre": pre, "post": post}}
        self.audit.write({"event": "POST_RETRIEVAL", "tool": "get_market_snapshot",
                          "status": status, "successful_count": len(observations), "decision": post})
        return result

    def history(self, symbol, period, interval, authorization):
        pre = self.governance.authorize_request(authorization, symbol, period, interval)
        self.audit.write({"event": "PRE_RETRIEVAL", "tool": "get_market_history",
                          "arguments": {"symbol": symbol, "period": period, "interval": interval}, "decision": pre})
        if not pre["accepted"]: raise PermissionError(f"Request rejected: {pre['gates']}")
        records = self.provider.get_history(symbol, period, interval)
        return {"symbol": symbol, "name": APPROVED_UNIVERSE[symbol], "period": period,
                "interval": interval, "provider_adapter": self.provider.name, "records": records}
'''
(PROJECT_ROOT / "audit.py").write_text(audit_py, encoding="utf-8")
(PROJECT_ROOT / "service.py").write_text(service_py, encoding="utf-8")
print("Service and audit layers written.")

Service and audit layers written.


## 7. MCP protocol layer

The server publishes four stable tools. It contains almost no business logic: calls are delegated to `MarketService`. The same tools work with the simulated adapter, Yahoo, or a future licensed provider.

In [7]:
# 7A. Write server.py. Environment variables select provider, audit path, and authorization.
server_py = r'''import os
from pathlib import Path
from mcp.server.fastmcp import FastMCP

from audit import AuditLog
from governance import GovernanceEngine
from providers import SimulatedProvider, YFinanceProvider
from service import MarketService

mcp = FastMCP("Governed Yahoo Market Server")
provider = YFinanceProvider() if os.getenv("MARKET_PROVIDER", "simulated") == "yahoo" else SimulatedProvider()
audit = AuditLog(Path(os.getenv("MARKET_AUDIT_PATH", "audit/events.jsonl")))
service = MarketService(provider, GovernanceEngine(os.getenv("MARKET_AUTHORIZATION", "RESEARCH-AUTHORIZED")), audit)

@mcp.tool()
def get_market_universe() -> dict:
    "Return the market symbols this server is authorized to retrieve."
    return service.universe()

@mcp.tool()
def get_quote(symbol: str, authorization: str) -> dict:
    "Retrieve and govern the latest available observation for an approved symbol."
    return service.quote(symbol, authorization)

@mcp.tool()
def get_market_snapshot(authorization: str) -> dict:
    "Retrieve a partial-failure-aware snapshot of the complete approved universe."
    return service.snapshot(authorization)

@mcp.tool()
def get_market_history(symbol: str, period: str, interval: str, authorization: str) -> dict:
    "Retrieve bounded OHLCV history for an approved symbol."
    return service.history(symbol, period, interval, authorization)

if __name__ == "__main__":
    mcp.run()  # local stdio transport; remote deployment is a separate decision
'''
(PROJECT_ROOT / "server.py").write_text(server_py, encoding="utf-8")
(PROJECT_ROOT / "__init__.py").write_text("", encoding="utf-8")
print(server_py)

import os
from pathlib import Path
from mcp.server.fastmcp import FastMCP

from audit import AuditLog
from governance import GovernanceEngine
from providers import SimulatedProvider, YFinanceProvider
from service import MarketService

mcp = FastMCP("Governed Yahoo Market Server")
provider = YFinanceProvider() if os.getenv("MARKET_PROVIDER", "simulated") == "yahoo" else SimulatedProvider()
audit = AuditLog(Path(os.getenv("MARKET_AUDIT_PATH", "audit/events.jsonl")))
service = MarketService(provider, GovernanceEngine(os.getenv("MARKET_AUTHORIZATION", "RESEARCH-AUTHORIZED")), audit)

@mcp.tool()
def get_market_universe() -> dict:
    "Return the market symbols this server is authorized to retrieve."
    return service.universe()

@mcp.tool()
def get_quote(symbol: str, authorization: str) -> dict:
    "Retrieve and govern the latest available observation for an approved symbol."
    return service.quote(symbol, authorization)

@mcp.tool()
def get_market_snapshot(authorization: str) -> dict:

In [8]:
# 7B. Make the generated modules importable in this runtime.
import sys, importlib
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()

from audit import AuditLog
from domain import APPROVED_UNIVERSE
from governance import GovernanceEngine
from providers import SimulatedProvider, YFinanceProvider
from service import MarketService

AUTHORIZATION = "RESEARCH-AUTHORIZED"
audit = AuditLog(VAULT_ROOT / "04_Audit" / "events.jsonl")
governance = GovernanceEngine(AUTHORIZATION)
service = MarketService(SimulatedProvider(), governance, audit)
service.universe()

{'symbols': {'^GSPC': 'S&P 500',
  '^DJI': 'Dow Jones Industrial Average',
  '^IXIC': 'Nasdaq Composite',
  '^FTSE': 'FTSE 100',
  '^N225': 'Nikkei 225'},
 'count': 5,
 'provider_adapter': 'deterministic-classroom-adapter'}

## 8. Inspect the discoverable MCP contracts

The model does not inspect Python internals. It receives names, descriptions, and JSON-compatible input schemas. The next cell asks the MCP framework for the actual registered tools without launching a blocking server.

In [9]:
# 8A. Load the generated server and inspect actual MCP tool metadata.
import server

tools = await server.mcp.list_tools()
tool_catalogue = [{"name": t.name, "description": t.description, "inputSchema": t.inputSchema} for t in tools]
tool_catalogue

[{'name': 'get_market_universe',
  'description': 'Return the market symbols this server is authorized to retrieve.',
  'inputSchema': {'properties': {},
   'title': 'get_market_universeArguments',
   'type': 'object'}},
 {'name': 'get_quote',
  'description': 'Retrieve and govern the latest available observation for an approved symbol.',
  'inputSchema': {'properties': {'symbol': {'title': 'Symbol',
     'type': 'string'},
    'authorization': {'title': 'Authorization', 'type': 'string'}},
   'required': ['symbol', 'authorization'],
   'title': 'get_quoteArguments',
   'type': 'object'}},
 {'name': 'get_market_snapshot',
  'description': 'Retrieve a partial-failure-aware snapshot of the complete approved universe.',
  'inputSchema': {'properties': {'authorization': {'title': 'Authorization',
     'type': 'string'}},
   'required': ['authorization'],
   'title': 'get_market_snapshotArguments',
   'type': 'object'}},
 {'name': 'get_market_history',
  'description': 'Retrieve bounded OHL

## 9. Prove refusal before success

A governed demonstration must show that unauthorized symbols and credentials fail. Successful calls alone do not establish governance.

In [10]:
# 9A. Deliberately failed authorization.
try:
    service.quote("^GSPC", "WRONG-TOKEN")
    raise AssertionError("The authorization gate did not reject the request")
except PermissionError as exc:
    print("✅ Expected authorization rejection:", exc)

✅ Expected authorization rejection: Request rejected: {'authorization': False, 'universe': True}


In [11]:
# 9B. Deliberately failed universe gate.
try:
    service.quote("UNAPPROVED", AUTHORIZATION)
    raise AssertionError("The universe gate did not reject the symbol")
except PermissionError as exc:
    print("✅ Expected universe rejection:", exc)

✅ Expected universe rejection: Request rejected: {'authorization': True, 'universe': False}


In [12]:
# 9C. Deliberately failed parameter gate.
try:
    service.history("^GSPC", "10y", "1m", AUTHORIZATION)
    raise AssertionError("The period gate did not reject the request")
except PermissionError as exc:
    print("✅ Expected period rejection:", exc)

✅ Expected period rejection: Request rejected: {'authorization': True, 'universe': True, 'period': False, 'interval': True}


## 10. Deterministic end-to-end execution

The default validation path uses simulated evidence so the architecture can be reproduced even when Yahoo is closed, unreachable, throttled, or changed.

In [13]:
# 10A. Governed quote and full snapshot.
import pandas as pd

quote_result = service.quote("^GSPC", AUTHORIZATION)
snapshot_result = service.snapshot(AUTHORIZATION)
assert snapshot_result["status"] == "ACCEPTED"
assert snapshot_result["successful_count"] == len(APPROVED_UNIVERSE)
pd.DataFrame(snapshot_result["observations"])

,symbol,name,value,change_pct,volume,observation_timestamp,retrieved_at,provider_adapter,source_family,classification
0,^GSPC,S&P 500,6319.53,0.31,None,2026-07-22T15:52:55.168867+00:00,2026-07-22T15:52:55.168867+00:00,deterministic-classroom-adapter,Synthetic classroom data,SIMULATED
1,^DJI,Dow Jones Industrial Average,45036.00,0.08,None,2026-07-22T15:52:55.168884+00:00,2026-07-22T15:52:55.168884+00:00,deterministic-classroom-adapter,Synthetic classroom data,SIMULATED
2,^IXIC,Nasdaq Composite,21142.80,0.68,None,2026-07-22T15:52:55.168893+00:00,2026-07-22T15:52:55.168893+00:00,deterministic-classroom-adapter,Synthetic classroom data,SIMULATED
3,^FTSE,FTSE 100,8980.20,-0.22,None,2026-07-22T15:52:55.168901+00:00,2026-07-22T15:52:55.168901+00:00,deterministic-classroom-adapter,Synthetic classroom data,SIMULATED
4,^N225,Nikkei 225,41354.75,-0.35,None,2026-07-22T15:52:55.168908+00:00,2026-07-22T15:52:55.168908+00:00,deterministic-classroom-adapter,Synthetic classroom data,SIMULATED


## 11. Optional Yahoo retrieval

Set `RUN_YAHOO = True` to make a real network attempt. The cell preserves partial failures and the conservative classification. A successful call demonstrates retrieval, **not** exchange-grade liveness or licensing.

In [15]:
# 11A. Optional Yahoo-origin snapshot through yfinance.
RUN_YAHOO = True

if RUN_YAHOO:
    yahoo_service = MarketService(YFinanceProvider(), governance, audit)
    yahoo_snapshot = yahoo_service.snapshot(AUTHORIZATION)
    print({k: yahoo_snapshot[k] for k in ("status", "requested_count", "successful_count", "complete")})
    display(pd.DataFrame(yahoo_snapshot["observations"]))
    if yahoo_snapshot["errors"]:
        display(pd.DataFrame(yahoo_snapshot["errors"]))
else:
    print("Yahoo call skipped. Set RUN_YAHOO=True when you intentionally want network retrieval.")

{'status': 'ACCEPTED', 'requested_count': 5, 'successful_count': 5, 'complete': True}


,symbol,name,value,change_pct,volume,observation_timestamp,retrieved_at,provider_adapter,source_family,classification
0,^GSPC,S&P 500,7521.2700,0.4859,0.0,2026-07-22T11:53:00-04:00,2026-07-22T15:53:20.211826+00:00,yfinance-unofficial-yahoo-adapter,Yahoo Finance,DELAYED_OR_LIVE_UNVERIFIED
1,^DJI,Dow Jones Industrial Average,52387.5000,0.6035,0.0,2026-07-22T11:53:00-04:00,2026-07-22T15:53:20.564161+00:00,yfinance-unofficial-yahoo-adapter,Yahoo Finance,DELAYED_OR_LIVE_UNVERIFIED
2,^IXIC,Nasdaq Composite,25840.0293,0.3598,0.0,2026-07-22T11:53:00-04:00,2026-07-22T15:53:20.948910+00:00,yfinance-unofficial-yahoo-adapter,Yahoo Finance,DELAYED_OR_LIVE_UNVERIFIED
3,^FTSE,FTSE 100,10721.4297,2.0801,0.0,2026-07-22T16:29:00+01:00,2026-07-22T15:53:21.719717+00:00,yfinance-unofficial-yahoo-adapter,Yahoo Finance,DELAYED_OR_LIVE_UNVERIFIED
4,^N225,Nikkei 225,66005.1562,2.4771,0.0,2026-07-22T15:29:00+09:00,2026-07-22T15:53:22.042401+00:00,yfinance-unofficial-yahoo-adapter,Yahoo Finance,DELAYED_OR_LIVE_UNVERIFIED


## 12. Dynamic Obsidian vault

The vault is not merely storage. Each report links to its predecessors and therefore creates an explicit memory chain. Facts, derived metrics, interpretation, classification, and governance evidence remain distinguishable.

In [16]:
# 12A. Write vault.py.
vault_py = r'''import csv, hashlib, json
from datetime import datetime, timezone
from pathlib import Path

class DynamicVault:
    def __init__(self, root):
        self.root = Path(root)
        for folder in ("00_Governance", "01_Snapshots", "02_Reports", "03_Synthesis", "04_Audit"):
            (self.root / folder).mkdir(parents=True, exist_ok=True)

    def reports(self):
        return sorted((self.root / "02_Reports").glob("Report_*.md"))

    def save_cycle(self, cycle, snapshot):
        if snapshot["status"] != "ACCEPTED":
            raise PermissionError("Only fully governed snapshots may be published")
        previous = self.reports()
        if len(previous) != cycle - 1:
            raise RuntimeError(f"Memory gate failed: expected {cycle-1} predecessors, found {len(previous)}")
        observations = snapshot["observations"]
        csv_path = self.root / "01_Snapshots" / "observations.csv"
        new = not csv_path.exists()
        with csv_path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["cycle", *observations[0].keys()])
            if new: writer.writeheader()
            for row in observations: writer.writerow({"cycle": cycle, **row})
        avg = sum(x["change_pct"] for x in observations) / len(observations)
        spread = max(x["change_pct"] for x in observations) - min(x["change_pct"] for x in observations)
        links = ", ".join(f"[[{p.stem}]]" for p in previous) or "None — first accepted state"
        rows = "\n".join(f'| {x["name"]} | {x["value"]:,.2f} | {x["change_pct"]:+.2f}% | {x["classification"]} |' for x in observations)
        text = (f"---\ncycle: {cycle}\ngovernance_status: ACCEPTED\n"
                f"classification: {observations[0]['classification']}\n---\n"
                f"# Report {cycle:02d}\n\n## Predecessor memory consulted\n{links}\n\n"
                f"## Observed evidence\n| Index | Value | Change | Classification |\n"
                f"|---|---:|---:|---|\n{rows}\n\n## Derived state\n"
                f"Cross-index average: **{avg:+.2f}%**. Dispersion: **{spread:.2f} percentage points**.\n\n"
                "## Interpretation\nThis paragraph is interpretation, not provider-supplied fact. "
                "It must be read with the source classification above.\n")
        path = self.root / "02_Reports" / f"Report_{cycle:02d}.md"
        path.write_text(text, encoding="utf-8")
        return path

    def synthesize(self):
        reports = self.reports()
        if not reports: raise RuntimeError("No reports exist")
        text = "# Path-Dependent Market Synthesis\n\n" +                "This synthesis is grounded in the complete accepted report chain. " +                "It preserves the distinction between stored observations and subsequent interpretation.\n\n## Lineage\n" +                "\n".join(f"- [[{p.stem}]]" for p in reports)
        path = self.root / "03_Synthesis" / "Market_Synthesis.md"
        path.write_text(text, encoding="utf-8")
        return path

    def manifest(self):
        items = []
        for path in sorted(self.root.rglob("*")):
            if path.is_file() and path.name != "manifest.json":
                items.append({"path": str(path.relative_to(self.root)),
                              "bytes": path.stat().st_size,
                              "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
        payload = {"schema": "yahoo-market-mcp-audit/1.0",
                   "created_at": datetime.now(timezone.utc).isoformat(), "files": items}
        path = self.root / "04_Audit" / "manifest.json"
        path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        return path

    def verify(self):
        manifest = json.loads((self.root / "04_Audit" / "manifest.json").read_text())
        failures = []
        for item in manifest["files"]:
            path = self.root / item["path"]
            if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
                failures.append(item["path"])
        return {"valid": not failures, "failures": failures, "file_count": len(manifest["files"])}
'''
(PROJECT_ROOT / "vault.py").write_text(vault_py, encoding="utf-8")
importlib.invalidate_caches()
from vault import DynamicVault
print("Dynamic vault layer written.")

Dynamic vault layer written.


## 13. Agentic orchestration and the clock

The orchestrator asks for a snapshot, checks its status, retrieves predecessor memory, writes the report, and waits. MCP itself does not initiate the next cycle. The demonstration below runs immediately; a real classroom performance can set `interval_seconds=60`.

In [17]:
# 13A. Run five governed cycles into a fresh vault.
import shutil, time

# Start with fresh market artifacts while preserving the refusal evidence
# already written to 04_Audit/events.jsonl.
for folder in ("01_Snapshots", "02_Reports", "03_Synthesis"):
    target = VAULT_ROOT / folder
    if target.exists():
        shutil.rmtree(target)
vault = DynamicVault(VAULT_ROOT)

run_events = []
for cycle in range(1, 6):
    snapshot = service.snapshot(AUTHORIZATION)
    report = vault.save_cycle(cycle, snapshot)
    run_events.append({"cycle": cycle, "status": snapshot["status"],
                       "predecessors": cycle - 1, "report": report.name})
    # time.sleep(60)  # uncomment for a real one-minute cadence

pd.DataFrame(run_events)

,cycle,status,predecessors,report
0,1,ACCEPTED,0,Report_01.md
1,2,ACCEPTED,1,Report_02.md
2,3,ACCEPTED,2,Report_03.md
3,4,ACCEPTED,3,Report_04.md
4,5,ACCEPTED,4,Report_05.md


In [18]:
# 13B. Inspect the fifth report's explicit predecessor memory.
fifth_report = vault.reports()[-1]
print(fifth_report.read_text(encoding="utf-8"))

---
cycle: 5
governance_status: ACCEPTED
classification: SIMULATED
---
# Report 05

## Predecessor memory consulted
[[Report_01]], [[Report_02]], [[Report_03]], [[Report_04]]

## Observed evidence
| Index | Value | Change | Classification |
|---|---:|---:|---|
| S&P 500 | 6,319.53 | +0.31% | SIMULATED |
| Dow Jones Industrial Average | 45,036.00 | +0.08% | SIMULATED |
| Nasdaq Composite | 21,142.80 | +0.68% | SIMULATED |
| FTSE 100 | 8,980.20 | -0.22% | SIMULATED |
| Nikkei 225 | 41,354.75 | -0.35% | SIMULATED |

## Derived state
Cross-index average: **+0.10%**. Dispersion: **1.03 percentage points**.

## Interpretation
This paragraph is interpretation, not provider-supplied fact. It must be read with the source classification above.



In [19]:
# 13C. Produce the synthesis and machine-readable manifest.
synthesis_path = vault.synthesize()
manifest_path = vault.manifest()
verification = vault.verify()
assert verification["valid"]
print(synthesis_path.read_text(encoding="utf-8"))
print("\nIntegrity:", verification)

# Path-Dependent Market Synthesis

This synthesis is grounded in the complete accepted report chain. It preserves the distinction between stored observations and subsequent interpretation.

## Lineage
- [[Report_01]]
- [[Report_02]]
- [[Report_03]]
- [[Report_04]]
- [[Report_05]]

Integrity: {'valid': True, 'failures': [], 'file_count': 8}


## 14. Audit inspection

The audit stream records requests and decisions. The manifest records artifact identity. Together they answer different questions: *what happened and why?* versus *are these still the same bytes?*

In [20]:
# 14A. Inspect audit events and the manifest.
import json

events_path = VAULT_ROOT / "04_Audit" / "events.jsonl"
events = [json.loads(line) for line in events_path.read_text().splitlines()]
audit_table = pd.DataFrame([{"event": e.get("event"), "tool": e.get("tool"),
                            "status": e.get("status"), "recorded_at": e.get("recorded_at")} for e in events]).tail(12)
print(audit_table.to_string(index=False))
json.loads(manifest_path.read_text())

         event                tool   status                      recorded_at
 PRE_RETRIEVAL get_market_snapshot     None 2026-07-22T15:53:18.527024+00:00
POST_RETRIEVAL get_market_snapshot ACCEPTED 2026-07-22T15:53:22.042505+00:00
 PRE_RETRIEVAL get_market_snapshot     None 2026-07-22T15:53:36.822559+00:00
POST_RETRIEVAL get_market_snapshot ACCEPTED 2026-07-22T15:53:36.822909+00:00
 PRE_RETRIEVAL get_market_snapshot     None 2026-07-22T15:53:36.824232+00:00
POST_RETRIEVAL get_market_snapshot ACCEPTED 2026-07-22T15:53:36.824467+00:00
 PRE_RETRIEVAL get_market_snapshot     None 2026-07-22T15:53:36.825472+00:00
POST_RETRIEVAL get_market_snapshot ACCEPTED 2026-07-22T15:53:36.825754+00:00
 PRE_RETRIEVAL get_market_snapshot     None 2026-07-22T15:53:36.826315+00:00
POST_RETRIEVAL get_market_snapshot ACCEPTED 2026-07-22T15:53:36.826458+00:00
 PRE_RETRIEVAL get_market_snapshot     None 2026-07-22T15:53:36.826946+00:00
POST_RETRIEVAL get_market_snapshot ACCEPTED 2026-07-22T15:53:36.827095+00:00

{'schema': 'yahoo-market-mcp-audit/1.0',
 'created_at': '2026-07-22T15:53:45.090300+00:00',
 'files': [{'path': '01_Snapshots/observations.csv',
   'bytes': 4399,
   'sha256': '073414b61ba0f77777524694e25efa1985db5739e3df1911ae6ab47e62c11850'},
  {'path': '02_Reports/Report_01.md',
   'bytes': 715,
   'sha256': '8390da2a111112d3c78cb557d35999988a8d113861b49431a125359f57914ece'},
  {'path': '02_Reports/Report_02.md',
   'bytes': 699,
   'sha256': '44f03a122e8e41f6b293e2f096f72d0602c25ad3c756cfdf921cd578f494ef15'},
  {'path': '02_Reports/Report_03.md',
   'bytes': 714,
   'sha256': 'fd7d6beb0ef461f025b0ee8d9189f17ae33c9964403ecab68a1bfcdd1d703e96'},
  {'path': '02_Reports/Report_04.md',
   'bytes': 729,
   'sha256': '8b13160c1516ced7483d414385f4a7553f248e79bca30ebe46cd22b96468ebca'},
  {'path': '02_Reports/Report_05.md',
   'bytes': 744,
   'sha256': 'd0c3e239f5936aa59246fdbe09b2537a98f7e6134bb1a85716e495fe72381bab'},
  {'path': '03_Synthesis/Market_Synthesis.md',
   'bytes': 280,
   'sh

## 15. Executable integrity and lineage tests

These tests convert the paper's governance claims into falsifiable checks.

In [21]:
# 15A. Validate contracts, controls, memory, classification, and integrity.
assert set(APPROVED_UNIVERSE) == {"^GSPC", "^DJI", "^IXIC", "^FTSE", "^N225"}
assert {t["name"] for t in tool_catalogue} == {"get_market_universe", "get_quote", "get_market_snapshot", "get_market_history"}
assert len(vault.reports()) == 5
assert "None — first accepted state" in vault.reports()[0].read_text()
assert "[[Report_04]]" in vault.reports()[4].read_text()
assert all("classification: SIMULATED" in p.read_text() for p in vault.reports())
assert any(e.get("decision", {}).get("accepted") is False for e in events)
assert vault.verify()["valid"] is True
print("✅ Tool, authorization, universe, parameter, lineage, classification, and integrity tests passed.")

✅ Tool, authorization, universe, parameter, lineage, classification, and integrity tests passed.


## 16. Run the MCP server locally

Colab is well suited to constructing and inspecting the server, but it is not a persistent production host. The following command is intentionally not launched inside the notebook because an stdio server blocks while waiting for a client.

```bash
cd /content/Yahoo_Finance_MCP_Case/yahoo_market_mcp
MARKET_PROVIDER=yahoo MARKET_AUTHORIZATION=RESEARCH-AUTHORIZED python server.py
```

For remote use, deploy an authenticated HTTPS endpoint with a supported MCP transport. Do not expose a Colab tunnel as a production market-data service.

In [22]:
# 16A. Show the exact local launch command for this runtime.
launch_command = f'cd "{PROJECT_ROOT}" && MARKET_PROVIDER=yahoo MARKET_AUTHORIZATION=RESEARCH-AUTHORIZED python server.py'
print(launch_command)

cd "/content/Yahoo_Finance_MCP_Case/yahoo_market_mcp" && MARKET_PROVIDER=yahoo MARKET_AUTHORIZATION=RESEARCH-AUTHORIZED python server.py


## 17. Production migration checklist

Before any production use, replace or strengthen:

- `yfinance` with a licensed provider appropriate to the use case;
- Colab with a persistent, monitored runtime;
- static tokens with OAuth or equivalent scoped authentication;
- simple timestamp presence with exchange calendars and explicit staleness thresholds;
- sequential calls with bounded concurrency, retry budgets, backoff, and circuit breakers;
- local files with durable transactional persistence and retention controls;
- research-only tools with clear separation from any order authority;
- basic logs with centralized telemetry, alerting, privacy, and incident response;
- implicit deployment trust with TLS, secret management, rate limits, and egress controls.

The stable MCP contract can remain while the provider and infrastructure change.

## 18. Export the complete evidence packet

The ZIP contains the modular server source, dynamic Obsidian vault, reports, audit events, and manifest. The notebook itself is the transparent reconstruction layer.

In [24]:
# 18A. Export a portable ZIP and print all deliverables.
import shutil

zip_path = shutil.make_archive(str(WORK_ROOT / "Yahoo_Finance_MCP_Audit_Packet"), "zip", WORK_ROOT)
print({
    "notebook_work_root": str(WORK_ROOT),
    "server_source": str(PROJECT_ROOT),
    "obsidian_vault": str(VAULT_ROOT),
    "audit_packet": zip_path,
    "integrity": vault.verify(),
})

{'notebook_work_root': '/content/Yahoo_Finance_MCP_Case', 'server_source': '/content/Yahoo_Finance_MCP_Case/yahoo_market_mcp', 'obsidian_vault': '/content/Yahoo_Finance_MCP_Case/Yahoo_Market_Obsidian_Vault', 'audit_packet': '/content/Yahoo_Finance_MCP_Case/Yahoo_Finance_MCP_Audit_Packet.zip', 'integrity': {'valid': True, 'failures': [], 'file_count': 8}}


## Conclusion

The agent depends on a governed market-information contract—not Yahoo's page structure, a particular package, or an unexplained number returned at an unknown time.

The complete role separation is:

- **MCP:** discoverable bounded capabilities;
- **Yahoo / provider:** external observations;
- **orchestrator:** time, recurrence, stopping, and retry policy;
- **dynamic vault:** persistent path-dependent memory;
- **model:** interpretation over accepted evidence;
- **governance:** permission, constraints, qualification, and refusal;
- **audit layer:** accountability, lineage, and integrity.

Official technical references used in the companion paper:

- OpenAI Apps SDK — MCP server concepts: https://developers.openai.com/apps-sdk/concepts/mcp-server
- OpenAI Apps SDK — build an MCP server: https://developers.openai.com/apps-sdk/build/mcp-server
- OpenAI API — MCP and connectors: https://developers.openai.com/api/docs/guides/tools-connectors-mcp
- Model Context Protocol specification: https://modelcontextprotocol.io/specification
- `yfinance` documentation: https://ranaroussi.github.io/yfinance/

This notebook is an educational reference architecture. It does not establish data licensing, exchange-grade timeliness, fitness for trading, or investment advice.